- **Name:** 20.2_streaming_bad_data
- **Author:** Shamas Imran
- **Desciption:** Handling schema mismatch in streaming data
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Demonstrated bad-data handling                                                  
-->

In [ ]:
# ------------------------------
# 1) Paths (your folders)
# ------------------------------
rootPath = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/"
masterPath = rootPath + "spark-streaming/"
inputPath = masterPath + "csv_input"
checkpointPath = masterPath + "checkpoints/csv_query"
outputPath = masterPath + "csv_output"
badRecordsPath = masterPath + "baddata_output"

In [ ]:
# Check if folder exists before deleting
if mssparkutils.fs.exists(masterPath):
    mssparkutils.fs.rm(masterPath, True)  # True = recursive delete

In [ ]:
# Create directories
import os

os.makedirs(masterPath, exist_ok=True)
os.makedirs(inputPath, exist_ok=True)
os.makedirs(checkpointPath, exist_ok=True)
os.makedirs(outputPath, exist_ok=True)
os.makedirs(badRecordsPath, exist_ok=True)

In [ ]:
import pandas as pd
lakehouse_folder = inputPath

# Define data (with intentional invalid rows)
batch03_data = {
    "id": [1, 2, 3, None, 4, 5, 6, 7, 8],
    "name": ["John", "Jane", "Bob", "MissingID", "Alice", "Charlie", None, "David", "Eva"],
    "score": [85, 92, 78, 88, "abc", 95, 88, 82, 90],
    "event_time": [
        "2025-08-18 10:00:00",
        "2025-08-18 10:05:00",
        "2025-08-18 10:10:00",
        "2025-08-18 10:15:00",  # invalid id
        "2025-08-18 10:20:00",  # invalid score
        "not_a_timestamp",      # invalid event_time
        "2025-08-18 10:30:00",  # invalid name
        "2025-08-18 10:35:00",
        "2025-08-18 10:40:00"
    ]
}

batch03_df = pd.DataFrame(batch03_data)
batch03_path = f"{lakehouse_folder}/batch03_students_invalid.csv"
batch03_df["id"] = batch03_df["id"].astype("Int64")
batch03_df.to_csv(batch03_path, index=False)

print(batch03_df)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F

# ------------------------------
# 1) Read all as strings
# ------------------------------
raw_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("score", StringType(), True),
    StructField("event_time", StringType(), True)
])

df_raw = (
    spark.readStream
        .option("header", "true")
        .schema(raw_schema)
        .csv(inputPath)
)

# ------------------------------
# 2) Cast columns to proper types
# ------------------------------
df_casted = (
    df_raw
        .withColumn("id_int", F.col("id").cast("int"))
        .withColumn("score_int", F.col("score").cast("int"))
        .withColumn("event_time_ts", F.to_timestamp("event_time"))
)

# ------------------------------
# 3) Define valid vs invalid logic
# ------------------------------

# Valid = all properly casted and non-null
df_valid = df_casted.filter(
    "id_int IS NOT NULL AND name IS NOT NULL AND score_int IS NOT NULL AND event_time_ts IS NOT NULL"
)

# Invalid = anything that fails validation
df_invalid = df_casted.filter(
    "id_int IS NULL OR name IS NULL OR score_int IS NULL OR event_time_ts IS NULL"
)

# ------------------------------
# 4) Write valid and invalid outputs
# ------------------------------
query_valid = (
    df_valid
        .select("id_int", "name", "score_int", "event_time_ts")
        .writeStream
        .format("csv")
        .option("path", outputPath)
        .option("checkpointLocation", checkpointPath + "/valid")
        .outputMode("append")
        .trigger(once=True)
        .start()
)

query_invalid = (
    df_invalid
        .select("id", "name", "score", "event_time")
        .writeStream
        .format("csv")
        .option("path", badRecordsPath)
        .option("checkpointLocation", checkpointPath + "/invalid")
        .outputMode("append")
        .trigger(once=True)
        .start()
)

query_valid.awaitTermination()
query_invalid.awaitTermination()

print("✅ Streaming complete — valid and invalid data written correctly.")

In [ ]:
'''
id,name,score,event_time
1,John,85,2025-08-18 10:00:00
2,Jane,92,2025-08-18 10:05:00
3,Bob,78,2025-08-18 10:10:00
,MissingID,88,2025-08-18 10:15:00           # invalid: id is NULL
4,Alice,abc,2025-08-18 10:20:00              # invalid: score is not integer
5,Charlie,95,not_a_timestamp                  # invalid: event_time malformed
6,,88,2025-08-18 10:30:00                     # invalid: name is NULL (optional)
7,David,82,2025-08-18 10:35:00
8,Eva,90,2025-08-18 10:40:00
'''